In [ ]:
#image segmentation with my prtraind network on image in data set
#Import libriries
import segmentation_models_pytorch as smp
from segmentation_models_pytorch import Unet
# Use smp.Unet to explicitly call the Unet class from the library
model = smp.Unet(encoder_name="efficientnet-b7",  # choose encoder, e.g., mobilenet_v2 or efficientnet-b7
                encoder_weights="imagenet",     # use `imagenet` pre-trained weights for encoder initialization
                in_channels=3,                  # model input channels (1 for gray-scale images, 3 for RGB, etc.)
                classes=2)                      # model output channels (number of classes in your dataset)

model_path = "/content/drive/MyDrive/Thesis/POM-IMG/pom_seg_img_code/unet_model_improve_efficientnet-b7.pth"  # Replace with your model path

# Load the model's state_dict
model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
model.eval()  # Ensure the model is in evaluation mode

# Define transformation for resizing
resize_transform = transforms.Compose([
    transforms.Resize((160, 256)),
    transforms.ToTensor()
])

# Function to generate a binary mask and combine it with the original image
def generate_combined_image(model, image_path):
    # Open the image
    image = Image.open(image_path).convert("RGB")
    input_tensor = resize_transform(image).unsqueeze(0)

    # Generate mask
    with torch.no_grad():
        output = model(input_tensor)  # Model output
    binary_mask = (output.squeeze(0).argmax(0).numpy() > 0).astype(np.uint8)  # Binary mask: 1 for segmented, 0 otherwise

    # Resize the mask to the resized image shape
    mask_resized = binary_mask * 255  # Keep the binary mask as is

    # Combine the mask with the resized original image
    image_resized = np.array(resize_transform(image).permute(1, 2, 0) * 255).astype(np.uint8)  # Resized image as numpy array
    combined_image = np.where(mask_resized[:, :, None] > 0, image_resized, 0).astype(np.uint8)  # Apply the mask

    # Convert back to PIL image for saving
    combined_image = Image.fromarray(combined_image)
    return combined_image

# Process all images to generate combined images
parent_folder = "/content/drive/MyDrive/Thesis/POM-IMG/Data-14.10.2024"  # Replace with your parent folder path
for root, dirs, files in os.walk(parent_folder):
    if "Crop_RGB" in dirs:  # Ensure it matches the actual subfolder name
        crop_rgb_folder = os.path.join(root, "Crop_RGB")
        semgnt_folder = os.path.join(root, "semgnt_img")
        os.makedirs(semgnt_folder, exist_ok=True)

        for file_name in os.listdir(crop_rgb_folder):
            if file_name.lower().endswith((".png", ".jpg", ".jpeg")):
                image_path = os.path.join(crop_rgb_folder, file_name)
                combined_image = generate_combined_image(model, image_path)

                # Save the combined image
                output_path = os.path.join(semgnt_folder, file_name)
                combined_image.save(output_path)

print("Combined images generated and saved in the respective 'semgnt_img' folders.")